<a href="https://colab.research.google.com/github/Saransh-Basu-01/Advanced-Pytorch/blob/main/Monitoring_and_Debugging_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn

# Sample input (batch of 1 image, 1 channel, 28x28)
dummy_input = torch.randn(1, 1, 28, 28)

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5) # Output: (1, 10, 24, 24)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2) # Output: (1, 10, 12, 12)
        # Mistake: Incorrectly calculated or hardcoded in_features
        self.fc1 = nn.Linear(10 * 10 * 10, 50) # Expects 1000 features

    def forward(self, x):
        print(f"Input shape: {x.shape}")
        x = self.pool(self.relu(self.conv1(x)))
        print(f"Shape after conv/pool: {x.shape}")
        # Incorrect flatten attempt
        # x = x.view(-1, 10 * 10 * 10) # This will cause a runtime error if run
        # Correct flatten
        x = x.view(x.size(0), -1) # Flatten all dimensions except batch
        print(f"Shape after flattening: {x.shape}")
        # Now x shape is [1, 1440] because 10 * 12 * 12 = 1440
        # The fc1 layer below expects 1000 features, causing a mismatch!
        try:
            x = self.fc1(x)
        except RuntimeError as e:
            print(f"\nError occurred: {e}")
            print(f"Input shape to fc1: {x.shape}")
            print(f"fc1 expects input features: {self.fc1.in_features}")

# Instantiate and run
model = SimpleNet()
model(dummy_input)


Input shape: torch.Size([1, 1, 28, 28])
Shape after conv/pool: torch.Size([1, 10, 12, 12])
Shape after flattening: torch.Size([1, 1440])

Error occurred: mat1 and mat2 shapes cannot be multiplied (1x1440 and 1000x50)
Input shape to fc1: torch.Size([1, 1440])
fc1 expects input features: 1000


# using PDB

In [3]:
import torch
import torch.nn as nn
import pdb

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 20)
        self.activation = nn.ReLU()
        # Potential mistake: Input size doesn't match layer1 output
        self.layer2 = nn.Linear(25, 5) # ERROR is likely here (25 != 20)

    def forward(self, x):
        print(f"Initial shape: {x.shape}")
        x = self.layer1(x)
        print(f"After layer1: {x.shape}")
        x = self.activation(x)
        print(f"After activation: {x.shape}")
        # Let's debug before layer2
        pdb.set_trace()
        # This line will likely cause a runtime error
        x = self.layer2(x)
        print(f"After layer2: {x.shape}")
        return x

# Example usage
net = SimpleNet()
# Create a dummy input tensor
input_tensor = torch.randn(32, 10) # Batch of 32, features=10
output = net(input_tensor)


Initial shape: torch.Size([32, 10])
After layer1: torch.Size([32, 20])
After activation: torch.Size([32, 20])
> /tmp/ipykernel_4489/3589028214.py(22)forward()
     20         pdb.set_trace()
     21         # This line will likely cause a runtime error
---> 22         x = self.layer2(x)
     23         print(f"After layer2: {x.shape}")
     24         return x

ipdb> x.shape
torch.Size([32, 20])
ipdb> self.layer2
Linear(in_features=25, out_features=5, bias=True)
ipdb> self.layer2.in_features
25
--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user


RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x20 and 25x5)